<a href="https://colab.research.google.com/github/bsyeol/26AI_project/blob/main/notebooks/model_B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# =====================================================================
# [B0-0] 패키지 설치
# - TabPFN 패키지를 설치합니다. 런타임을 새로 시작할 때마다 한 번 실행합니다.
# - 런타임 유형이 GPU인지 먼저 확인하세요 (CPU는 5,000행까지만 허용).
# =====================================================================
!pip install -q tabpfn

In [17]:
import tabpfn
print(tabpfn.__version__)
##버전 고정용

9.2.0


In [5]:
# =====================================================================
# [B0-1] 공통 데이터 불러오기
# - 데이터 정리 노트북이 저장한 파일을 읽습니다. 각자 분할하지 않습니다.
# - settings.json의 값(SEEDS, BASE_FEATS, LABEL 등)을 변수로 만듭니다.
# - MD5로 모든 팀원이 같은 파일을 쓰는지 확인합니다. 다르면 여기서 멈춥니다.
# - 원본에 없는 feature 두 개(log_amount, dow)를 만듭니다.
# =====================================================================
from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib, time
import numpy as np
import pandas as pd

MID_DIR = '/content/drive/MyDrive/26-2_AI_project/processed/mid'
info = json.load(open(f'{MID_DIR}/split_info.json', encoding='utf-8'))
S    = json.load(open(f'{MID_DIR}/settings.json', encoding='utf-8'))
globals().update(S)

csv_path = f'{MID_DIR}/mid_v1.csv'
assert hashlib.md5(open(csv_path, 'rb').read()).hexdigest() == info['md5'], '파일이 다릅니다'

mid = pd.read_csv(csv_path, encoding='utf-8-sig',
                  dtype={USER: str, '입금계좌일련번호': str}, parse_dates=['date'])
mid['log_amount'] = np.log1p(mid['거래금액'])
mid['dow'] = mid['date'].dt.dayofweek

CONTEXT = {s: np.load(f'{MID_DIR}/context_seed{s}.npy') for s in SEEDS}

print(mid['split'].value_counts().to_dict(), '| 기대값:', info['rows'])
print('context:', {s: len(v) for s, v in CONTEXT.items()})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'train': 229653, 'test': 50264, 'val': 49404} | 기대값: {'train': 229653, 'test': 50264, 'val': 49404}
context: {0: 9996, 1: 9996, 2: 9996}


In [6]:
# =====================================================================
# [B0-2] 입력 feature 확인
# - B0이 쓰는 feature 7개의 범위와 결측을 확인합니다.
# - 라벨 파생 컬럼(DROP_COLS)이 섞여 있지 않은지 검사합니다.
# - context의 이상거래 비율이 설정대로(약 1/21) 맞는지 봅니다.
# =====================================================================
assert not set(BASE_FEATS) & set(DROP_COLS), '라벨 파생 컬럼이 입력에 있습니다'

print(mid[BASE_FEATS].describe().T[['min', 'max', 'mean']].round(2))
print('결측:', mid[BASE_FEATS].isna().sum().to_dict())

for s, idx in CONTEXT.items():
    y = mid.loc[idx, LABEL]
    print(f'seed {s}: context {len(idx)}행, 이상거래 {int(y.sum())}건 ({y.mean():.2%})')

               min     max    mean
log_amount    0.69   20.03   11.10
거래시간대         0.00   21.00   12.86
dow           0.00    6.00    3.00
매체구분          1.00    7.00    3.19
자금구분          0.00    4.00    0.01
출금금융회사일련번호  102.00  161.00  146.19
입금금융회사일련번호  101.00  160.00  147.55
결측: {'log_amount': 0, '거래시간대': 0, 'dow': 0, '매체구분': 0, '자금구분': 0, '출금금융회사일련번호': 0, '입금금융회사일련번호': 0}
seed 0: context 9996행, 이상거래 476건 (4.76%)
seed 1: context 9996행, 이상거래 476건 (4.76%)
seed 2: context 9996행, 이상거래 476건 (4.76%)


In [7]:
# =====================================================================
# [B0-3] TabPFN 준비
# - 보안 비밀에 저장한 TABPFN_TOKEN을 읽어 라이선스 인증을 합니다.
# - GPU가 잡히는지 확인합니다. 'cpu'가 나오면 런타임 유형을 바꾸세요.
# =====================================================================
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score, recall_score

from google.colab import userdata
os.environ['TABPFN_TOKEN'] = userdata.get('TABPFN_TOKEN')
from tabpfn import TabPFNClassifier

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')

device: cuda


In [12]:
# =====================================================================
# [B0-3-1] TabPFN 토큰 확인
# - 보안 비밀에서 토큰을 읽어 앞뒤 공백을 지우고 환경변수에 넣습니다.
# - 토큰 값은 출력하지 않고 길이와 상태만 확인합니다.
# =====================================================================
from google.colab import userdata
tok = userdata.get('TABPFN_TOKEN')
os.environ['TABPFN_TOKEN'] = tok.strip()

print('길이:', len(tok))
print('앞뒤 공백 있었음:', tok != tok.strip())
print('환경변수 설정됨:', os.environ.get('TABPFN_TOKEN') == tok.strip())

길이: 53
앞뒤 공백 있었음: False
환경변수 설정됨: True


In [13]:
# =====================================================================
# [B0-4] 평가 함수와 TabPFN 실행 함수 (모든 모델 공통, 수정하지 않음)
# - evaluate: PR-AUC(주지표)를 val, test에서 계산합니다.
#   F1, Recall은 val에서 F1이 가장 높은 threshold를 찾아 test에 그대로 적용합니다.
#   (test로 threshold를 고르면 test 정보가 새므로 반드시 val에서 고릅니다.)
# - run_tabpfn: seed별 고정 context로 fit(= context 전달)하고 val, test를 예측합니다.
#   예측은 PRED_BATCH 행씩 나눠서 해서 GPU 메모리 부족을 막습니다.
#   모델끼리 다른 것은 feats(입력 컬럼 목록)뿐입니다.
# =====================================================================
def evaluate(p_val, y_val, p_test, y_test):
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    t = thr[np.argmax(f1[:-1])]
    pred = (p_test >= t).astype(int)
    return {
        'val_pr_auc':  average_precision_score(y_val, p_val),
        'test_pr_auc': average_precision_score(y_test, p_test),
        'test_f1':     f1_score(y_test, pred),
        'test_recall': recall_score(y_test, pred),
        'threshold':   float(t),
    }

def run_tabpfn(feats, seed):
    ctx = CONTEXT[seed]
    clf = TabPFNClassifier(random_state=seed)
    clf.fit(mid.loc[ctx, feats].to_numpy(np.float32), mid.loc[ctx, LABEL].to_numpy())

    def proba(part):
        X = mid.loc[mid['split'] == part, feats].to_numpy(np.float32)
        return np.concatenate([clf.predict_proba(X[i:i + PRED_BATCH])[:, 1]
                               for i in range(0, len(X), PRED_BATCH)])

    y_val  = mid.loc[mid['split'] == 'val',  LABEL].to_numpy()
    y_test = mid.loc[mid['split'] == 'test', LABEL].to_numpy()
    return evaluate(proba('val'), y_val, proba('test'), y_test)

In [15]:
# =====================================================================
# [B0-5] B0 실행
# - 현재 거래 feature 7개(BASE_FEATS)만으로 seed마다 TabPFN을 실행합니다.
# - 처음에는 SEEDS = [0]으로 한 번 돌려 시간과 동작을 확인하는 것을 권합니다.
# - 참고용 기준선: 무작위로 찍었을 때의 PR-AUC는 test 이상거래 비율(약 0.63%)과 같습니다.
# =====================================================================
results = []
for seed in SEEDS:
    t0 = time.time()
    r = run_tabpfn(BASE_FEATS, seed)
    results.append({'model': 'B0', 'seed': seed, **r})
    print(f'B0 seed {seed}: val PR-AUC {r["val_pr_auc"]:.4f} | test PR-AUC {r["test_pr_auc"]:.4f} '
          f'| F1 {r["test_f1"]:.4f} | Recall {r["test_recall"]:.4f}  ({time.time() - t0:.0f}s)')

base_rate = mid.loc[mid['split'] == 'test', LABEL].mean()
print(f'\n무작위 기준 PR-AUC (test 이상거래 비율): {base_rate:.4f}')

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

B0 seed 0: val PR-AUC 0.5238 | test PR-AUC 0.5735 | F1 0.6080 | Recall 0.9842  (185s)
B0 seed 1: val PR-AUC 0.5271 | test PR-AUC 0.6178 | F1 0.5996 | Recall 1.0000  (173s)
B0 seed 2: val PR-AUC 0.5569 | test PR-AUC 0.5977 | F1 0.6006 | Recall 0.9968  (173s)

무작위 기준 PR-AUC (test 이상거래 비율): 0.0063


In [16]:
# =====================================================================
# [B0-6] 결과 정리와 저장
# - seed별 결과와 평균 ± 표준편차를 출력합니다.
# - processed/mid/results_B0.csv로 저장해 E4 등 다른 모델 결과와 합칠 수 있게 합니다.
# =====================================================================
res = pd.DataFrame(results)
print(res.round(4).to_string(index=False))

summary = res[['val_pr_auc', 'test_pr_auc', 'test_f1', 'test_recall']].agg(['mean', 'std']).round(4)
print(summary)

res.to_csv(f'{MID_DIR}/results_B0.csv', index=False)
print('저장:', f'{MID_DIR}/results_B0.csv')

model  seed  val_pr_auc  test_pr_auc  test_f1  test_recall  threshold
   B0     0      0.5238       0.5735   0.6080       0.9842     0.5428
   B0     1      0.5271       0.6178   0.5996       1.0000     0.2675
   B0     2      0.5569       0.5977   0.6006       0.9968     0.4314
      val_pr_auc  test_pr_auc  test_f1  test_recall
mean      0.5359       0.5963   0.6027       0.9937
std       0.0182       0.0222   0.0046       0.0084
저장: /content/drive/MyDrive/26-2_AI_project/processed/mid/results_B0.csv
